In [ ]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_parquet("/Users/arjunprakashrao/Drive/projects/Polymarket-lab/cricket_arbitrage/internal_data/generated/dls_dataset.parquet")
df.head()

,match_id,season,venue,city,date,innings,team,opponent,batter,bowler,...,is_dot,is_six,is_boundary,is_wicket,phase,censored,target_total,final_total,team_id,opponent_team_id
0,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,YBK Jaiswal,C Wickramasinghe,...,1,0,0,0,powerplay,False,NaN,137,40,89
1,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,YBK Jaiswal,C Wickramasinghe,...,0,0,0,0,powerplay,False,NaN,137,40,89
2,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,Shubman Gill,C Wickramasinghe,...,1,0,0,0,powerplay,False,NaN,137,40,89
3,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,Shubman Gill,C Wickramasinghe,...,0,0,0,0,powerplay,False,NaN,137,40,89
4,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,YBK Jaiswal,C Wickramasinghe,...,0,0,0,0,powerplay,False,NaN,137,40,89


In [5]:
df.columns

Index(['match_id', 'season', 'venue', 'city', 'date', 'innings', 'team',
       'opponent', 'batter', 'bowler', 'toss_winner', 'toss_decision',
       'toss_win_match_team', 'over_number', 'ball_in_over', 'balls_remaining',
       'legal_balls_bowled', 'wickets_in_hand', 'current_score',
       'runs_remaining_target', 'batter_score', 'batter_balls_faced',
       'bowler_wickets_in_match', 'runs_off_ball', 'total_runs_ball', 'is_dot',
       'is_six', 'is_boundary', 'is_wicket', 'phase', 'censored',
       'target_total', 'final_total', 'team_id', 'opponent_team_id'],
      dtype='str')

In [6]:
df[df['match_id'] == '76b58556-7d1c-4']['phase'].unique()

<ArrowStringArray>
[]
Length: 0, dtype: str

In [10]:
import pandas as pd
import numpy as np
from scipy.stats import poisson, nbinom

def generate_cricket_advanced_report(df, batting_team, bowling_team):
    """
    Generates a Cricket Stats report using:
    - Negative Binomial for RUNS (if variance > mean)
    - Poisson for RUNS (if variance <= mean)
    - Poisson for WICKETS (standard)
    """
    
    print(f"ADVANCED REPORT: {batting_team} (Bat) vs {bowling_team} (Bowl)")
    print("="*100)

    # 1. Filter Data
    matchup_df = df[
        (df['team'] == batting_team) & 
        (df['opponent'] == bowling_team)
    ].copy()
    
    if matchup_df.empty:
        print(f"No data found for {batting_team} vs {bowling_team}")
        return

    # 2. Ensure Numeric Columns
    cols_to_convert = ['over_number', 'total_runs_ball', 'is_wicket', 'is_six', 'is_boundary']
    for col in cols_to_convert:
        if col in matchup_df.columns:
            matchup_df[col] = pd.to_numeric(matchup_df[col], errors='coerce').fillna(0)

    # 3. Handle Over Binning
    min_over = matchup_df['over_number'].min()
    if min_over == 0:
        bins = [0, 3, 6, 9, 12, 15, 18, 21]
        labels = ['Overs 0-2', 'Overs 3-5', 'Overs 6-8', 'Overs 9-11', 'Overs 12-14', 'Overs 15-17', 'Overs 18-20']
    else:
        bins = [1, 4, 7, 10, 13, 16, 19, 22]
        labels = ['Overs 1-3', 'Overs 4-6', 'Overs 7-9', 'Overs 10-12', 'Overs 13-15', 'Overs 16-18', 'Overs 19-20']
    
    matchup_df['over_bin'] = pd.cut(matchup_df['over_number'], bins=bins, labels=labels, right=False)

    # 4. Aggregate Data per Match & Interval
    grouped = matchup_df.groupby(['match_id', 'over_bin'], observed=False).agg({
        'total_runs_ball': 'sum',   
        'is_wicket': 'sum',         
        'is_six': 'sum',            
        'is_boundary': 'sum'        
    }).reset_index()

    # 5. Generate Stats and Report
    print(f"{'Interval':<15} | {'Matches':<8} | {'Exp Runs':<10} | {'Exp Wkts':<10} | {'Exp 6s':<10} | {'Reliability'}")
    print("-" * 100)

    for interval in labels:
        interval_data = grouped[grouped['over_bin'] == interval]
        
        if interval_data.empty:
            continue
            
        n_matches = interval_data['match_id'].nunique()
        
        # --- CALCULATE STATS ---
        runs_array = interval_data['total_runs_ball'].values
        
        mean_runs = runs_array.mean()
        var_runs = runs_array.var(ddof=1) if n_matches > 1 else 0 # Variance
        
        mean_wickets = interval_data['is_wicket'].mean()
        mean_sixes = interval_data['is_six'].mean()
        
        # Reliability Label
        if n_matches < 5: reliability = "LOW (Caution)"
        elif n_matches >= 20: reliability = "HIGH (Reliable)"
        else: reliability = "MEDIUM"

        print(f"{interval:<15} | {n_matches:<8} | {mean_runs:<10.2f} | {mean_wickets:<10.2f} | {mean_sixes:<10.2f} | {reliability}")
        
        # --- PREDICTION LOGIC ---
        if n_matches >= 3:
            # 1. RUNS MODEL SELECTION
            prob_high = 0
            prob_low = 0
            model_used = ""
            
            # CHECK OVERDISPERSION (Variance > Mean)
            if var_runs > mean_runs:
                # --- USE NEGATIVE BINOMIAL ---
                # Convert Mean/Var to n, p for Scipy
                # p = probability of success in each trial
                # n = number of successes
                # Formula: p = mean / variance
                #          n = mean^2 / (variance - mean)
                p_nb = mean_runs / var_runs
                n_nb = (mean_runs**2) / (var_runs - mean_runs)
                
                # Calculate probs
                prob_high = (1 - nbinom.cdf(25, n_nb, p_nb)) * 100
                prob_low = nbinom.cdf(14, n_nb, p_nb) * 100
                model_used = f"NegBin (Var={var_runs:.1f})"
                
            else:
                # --- USE POISSON (Standard) ---
                prob_high = (1 - poisson.cdf(25, mean_runs)) * 100
                prob_low = poisson.cdf(14, mean_runs) * 100
                model_used = "Poisson"

            # 2. WICKETS (Poisson)
            prob_wicket = (1 - poisson.cdf(0, mean_wickets)) * 100
            
            # 3. SIXES (Poisson)
            prob_six = (1 - poisson.cdf(0, mean_sixes)) * 100
            
            print(f"   [Runs Model: {model_used}]")
            print(f"   > Prob(Runs > 25): {prob_high:.1f}%   |   Prob(Runs < 15): {prob_low:.1f}%")
            print(f"   > Prob(Wicket):    {prob_wicket:.1f}%   |   Prob(Any Six):   {prob_six:.1f}%")
        else:
            print("   > Not enough data for reliable predictions.")
            
        print("-" * 100)

# Example Usage:
generate_cricket_advanced_report(df, 'England', 'Sri Lanka')


ADVANCED REPORT: England (Bat) vs Sri Lanka (Bowl)
Interval        | Matches  | Exp Runs   | Exp Wkts   | Exp 6s     | Reliability
----------------------------------------------------------------------------------------------------
Overs 0-2       | 17       | 20.00      | 1.00       | 0.24       | MEDIUM
   [Runs Model: NegBin (Var=77.6)]
   > Prob(Runs > 25): 24.0%   |   Prob(Runs < 15): 29.1%
   > Prob(Wicket):    63.2%   |   Prob(Any Six):   21.0%
----------------------------------------------------------------------------------------------------
Overs 3-5       | 17       | 25.53      | 0.65       | 0.65       | MEDIUM
   [Runs Model: NegBin (Var=77.3)]
   > Prob(Runs > 25): 46.3%   |   Prob(Runs < 15): 8.9%
   > Prob(Wicket):    47.6%   |   Prob(Any Six):   47.6%
----------------------------------------------------------------------------------------------------
Overs 6-8       | 17       | 21.00      | 0.41       | 0.71       | MEDIUM
   [Runs Model: NegBin (Var=46.6)]
   > Prob

In [8]:
df['team'].unique().tolist()

['India',
 'Sri Lanka',
 'West Indies',
 'South Africa',
 'Scotland',
 'Bermuda',
 'Australia',
 'Estonia',
 'Cyprus',
 'Rwanda',
 'Bahrain',
 'Portugal',
 'Spain',
 'Kuwait',
 'Hong Kong',
 'Ireland',
 'Swaziland',
 'Seychelles',
 'Belize',
 'United States of America',
 'England',
 'Indonesia',
 'Cambodia',
 'United Arab Emirates',
 'Denmark',
 'Italy',
 'Bangladesh',
 'Pakistan',
 'Finland',
 'Sweden',
 'Netherlands',
 'Japan',
 'Vanuatu',
 'Croatia',
 'Zimbabwe',
 'Namibia',
 'Mexico',
 'Cayman Islands',
 'Nepal',
 'New Zealand',
 'Saudi Arabia',
 'Guernsey',
 'Romania',
 'Czech Republic',
 'ICC World XI',
 'St Helena',
 'Botswana',
 'Philippines',
 'Gibraltar',
 'Bhutan',
 'Singapore',
 'Malaysia',
 'South Korea',
 'Bulgaria',
 'Panama',
 'Malta',
 'Switzerland',
 'Serbia',
 'Cook Islands',
 'Uganda',
 'Austria',
 'Germany',
 'Samoa',
 'Nigeria',
 'Eswatini',
 'Ghana',
 'Tanzania',
 'Oman',
 'Belgium',
 'Qatar',
 'Bahamas',
 'Argentina',
 'Sierra Leone',
 'Jersey',
 'Hungary',
 'Is

In [11]:
rows = []

df

,match_id,season,venue,city,date,innings,team,opponent,batter,bowler,...,is_dot,is_six,is_boundary,is_wicket,phase,censored,target_total,final_total,team_id,opponent_team_id
0,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,YBK Jaiswal,C Wickramasinghe,...,1,0,0,0,powerplay,False,NaN,137,40,89
1,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,YBK Jaiswal,C Wickramasinghe,...,0,0,0,0,powerplay,False,NaN,137,40,89
2,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,Shubman Gill,C Wickramasinghe,...,1,0,0,0,powerplay,False,NaN,137,40,89
3,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,Shubman Gill,C Wickramasinghe,...,0,0,0,0,powerplay,False,NaN,137,40,89
4,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,YBK Jaiswal,C Wickramasinghe,...,0,0,0,0,powerplay,False,NaN,137,40,89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
706199,7ff02753900ab9a,2019/20,Saxton Oval,Nelson,2019-11-05,2,England,New Zealand,TK Curran,TG Southee,...,1,0,0,0,death,False,181.0,166,25,66
706200,7ff02753900ab9a,2019/20,Saxton Oval,Nelson,2019-11-05,2,England,New Zealand,TK Curran,TG Southee,...,0,0,0,0,death,False,181.0,166,25,66
706201,7ff02753900ab9a,2019/20,Saxton Oval,Nelson,2019-11-05,2,England,New Zealand,S Mahmood,TG Southee,...,1,0,0,0,death,False,181.0,166,25,66
706202,7ff02753900ab9a,2019/20,Saxton Oval,Nelson,2019-11-05,2,England,New Zealand,S Mahmood,TG Southee,...,0,0,0,0,death,False,181.0,166,25,66
